# 🚕 Master EDA: NYC Taxi & Weather Analysis

Notebook ini adalah pusat analisis eksploratif untuk proyek RDV-Taxi. Di sini kita menggabungkan perspektif **Data Engineering** (kualitas data) dan **Data Analysis** (insight bisnis).

---

### 🎯 Tujuan Analisis:
1. Memastikan proses ELT berjalan benar (Data Quality).
2. Memahami perilaku penumpang taksi di NYC.
3. Menemukan hubungan antara kondisi cuaca dan penggunaan transportasi umum.

## 1. Setup & Environment

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
import os
import warnings

# Konfigurasi Visualisasi
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Koneksi ke Database
DB_PATH = os.path.join("..", "data", "final", "tlc.duckdb")
con = duckdb.connect(DB_PATH, read_only=True)
print(f"✅ Connected to: {os.path.abspath(DB_PATH)}")

## 2. Data Engineering Audit (Data Quality)
Langkah pertama DE adalah mengecek apakah pipeline pembersihan data bekerja dengan baik.

In [ ]:
# Bandingkan jumlah data sebelum dan sesudah dibersihkan
dq_query = """
SELECT 
    'Raw (tlc_raw)' as stage, COUNT(*) as count FROM tlc_raw
UNION ALL
SELECT 
    'Cleaned (tlc_cleaned)' as stage, COUNT(*) as count FROM tlc_cleaned
"""
dq_df = con.execute(dq_query).df()

plt.figure(figsize=(8, 5))
sns.barplot(data=dq_df, x='stage', y='count', palette='Set2')
plt.title('Data Retention: Raw vs Cleaned')
for i, v in enumerate(dq_df['count']):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.show()

## 3. Trip Metrics Distribution
Menganalisis statistik utama seperti tarif, jarak, dan durasi.

In [ ]:
# Ambil data sample untuk visualisasi cepat
df_trips = con.execute("SELECT * FROM fact_trips LIMIT 100000").df()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df_trips['fare_amount'], bins=50, ax=axes[0], color='green', kde=True)
axes[0].set_title('Distribution of Fare Amount')
axes[0].set_xlim(0, 100)

sns.histplot(df_trips['trip_distance'], bins=50, ax=axes[1], color='blue', kde=True)
axes[1].set_title('Distribution of Trip Distance')
axes[1].set_xlim(0, 20)

sns.histplot(df_trips['passenger_count'], bins=10, ax=axes[2], color='purple')
axes[2].set_title('Distribution of Passenger Count')

plt.tight_layout()
plt.show()

## 4. Temporal Patterns (Time Series)
Melihat fluktuasi permintaan berdasarkan waktu.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Hourly Demand
sns.countplot(data=df_trips, x='hour_of_day', ax=axes[0], palette='magma')
axes[0].set_title('Taxi Demand by Hour of Day')
axes[0].set_xlabel('Hour (0-23)')

# Daily Demand
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
sns.countplot(data=df_trips, x='day_of_week', order=day_order, ax=axes[1], palette='viridis')
axes[1].set_title('Taxi Demand by Day of Week')

plt.tight_layout()
plt.show()

## 5. Geospatial Insights
Analisis berdasarkan wilayah (Borough).

In [ ]:
borough_data = con.execute("""
    SELECT borough, COUNT(*) as trips, AVG(fare_amount) as avg_fare 
    FROM fact_trips f
    JOIN dim_location l ON f.pickup_location_key = l.location_key
    GROUP BY borough
    ORDER BY trips DESC
""").df()

plt.figure(figsize=(10, 6))
sns.barplot(data=borough_data, x='borough', y='trips', palette='rocket')
plt.title('Total Trips by Borough')
plt.xticks(rotation=45)
plt.show()

## 6. Weather Correlation
Bagaimana cuaca mempengaruhi tip dan permintaan?

In [ ]:
# Rata-rata tip berdasarkan kategori suhu
temp_tip = df_trips.groupby('temperature_category')['tip_percentage'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=temp_tip, x='temperature_category', y='tip_percentage', palette='coolwarm')
plt.title('Average Tip Percentage by Temperature Category')
plt.ylabel('Tip %')
plt.show()

In [ ]:
# Scatter plot Jarak vs Tarif dengan kondisi cuaca
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df_trips.sample(2000), x='trip_distance', y='fare_amount', 
                hue='weather_description', alpha=0.7)
plt.title('Correlation: Trip Distance vs Fare Amount by Weather')
plt.xlim(0, 25)
plt.ylim(0, 80)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## 7. Kesimpulan & Insight

Berdasarkan analisis di atas:
1. **Volume Data:** Kita berhasil menyaring sekitar ...% data yang dianggap anomali.
2. **Waktu Sibuk:** Permintaan memuncak pada jam ... dan paling rendah pada jam ...
3. **Faktor Cuaca:** Cuaca (Misal: Hujan) memiliki pengaruh sebesar ... terhadap rata-rata tip/tarif.

*-- Selesai --*

In [ ]:
con.close()
print("Database connection closed. EDA finished.")